# Resolution-Failure-Directed Extraction (RFDE)
## Neuro-Symbolic FOL Reasoning with Demand-Driven LLM Grounding

RFDE is a neuro-symbolic pipeline that uses SLD resolution failures as demand signals for LLM-based atomic fact extraction. When a proof attempt fails to resolve a predicate against the current knowledge base, it triggers a targeted LLM call to extract only the specific atomic fact needed.

In [ ]:
import subprocess, sys
def _pip(*a): subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *a])

_pip('rank_bm25==0.2.1')
if 'google.colab' not in sys.modules:
    _pip('requests==2.32.4', 'loguru==0.7.2')

print("✓ Dependencies installed")

In [ ]:
import json, os
GITHUB_DATA_URL = "https://raw.githubusercontent.com/AMGrobelnik/ai-invention-14efe0-resolution-failure-directed-extraction-d/main/round-1/experiment-1/demo/mini_demo_data.json"

def load_data():
    try:
        import urllib.request
        with urllib.request.urlopen(GITHUB_DATA_URL, timeout=10) as response:
            return json.loads(response.read().decode())
    except Exception:
        pass
    if os.path.exists("mini_demo_data.json"):
        with open("mini_demo_data.json") as f:
            return json.load(f)
    raise FileNotFoundError("Could not load mini_demo_data.json")

raw_data = load_data()
print(f"✓ Loaded {len(raw_data['datasets'][0]['examples'])} examples")

In [ ]:
# DEMO CONFIG (MINIMUM VALUES)
N_SYNTHETIC_TASKS = 1
SIMULATE_LLM_CALLS = True
MAX_DEPTH = 8

print(f"Tasks to run: {N_SYNTHETIC_TASKS}")

In [ ]:
import json, os, re, sys, time
from collections import defaultdict
from dataclasses import dataclass, field
from typing import Any

import resource
from loguru import logger
from rank_bm25 import BM25Okapi

logger.remove()
logger.add(sys.stdout, level="INFO")

RAM_BUDGET = 2 * 1024**3
resource.setrlimit(resource.RLIMIT_AS, (RAM_BUDGET * 2, RAM_BUDGET * 2))

# Global state
_total_cost_usd = 0.0
_total_llm_calls = 0

print("✓ Setup complete")

In [ ]:
def _llm_call_mock(prompt: str, max_tokens: int = 200) -> tuple[str, float]:
    global _total_cost_usd, _total_llm_calls
    if "mother" in prompt.lower():
        response = "ANSWER: yes\nCONFIDENCE: 1.0\nEVIDENCE: Alice is the mother of Bob."
    elif "father" in prompt.lower():
        response = "ANSWER: yes\nCONFIDENCE: 1.0\nEVIDENCE: Bob is the father of Charlie."
    else:
        response = "ANSWER: yes\nCONFIDENCE: 0.9\nEVIDENCE: document"
    time.sleep(0.02)
    cost = 0.0001
    _total_cost_usd += cost
    _total_llm_calls += 1
    return response, cost

print("✓ LLM mock function defined")

In [ ]:
@dataclass
class Term:
    functor: str
    args: list["Term"] = field(default_factory=list)

    def __repr__(self) -> str:
        if not self.args:
            return self.functor
        return f"{self.functor}({', '.join(repr(a) for a in self.args)})"

    def is_var(self) -> bool:
        return len(self.args) == 0 and self.functor[0].isupper()

@dataclass
class Clause:
    head: Term
    body: list[Term] = field(default_factory=list)
    confidence: float = 1.0
    source: str = "kb"

Subst = dict[str, Term]

def _walk(t: Term, subst: Subst) -> Term:
    while t.is_var() and t.functor in subst:
        t = subst[t.functor]
    return t

def _unify(t1: Term, t2: Term, subst: Subst) -> Subst | None:
    t1, t2 = _walk(t1, subst), _walk(t2, subst)
    if t1.is_var():
        if t1 == t2:
            return subst
        new = dict(subst)
        new[t1.functor] = t2
        return new
    if t2.is_var():
        new = dict(subst)
        new[t2.functor] = t1
        return new
    if t1.functor != t2.functor or len(t1.args) != len(t2.args):
        return None
    s = subst
    for a1, a2 in zip(t1.args, t2.args):
        s = _unify(a1, a2, s)
        if s is None:
            return None
    return s

def _apply_subst(t: Term, subst: Subst) -> Term:
    t = _walk(t, subst)
    if t.is_var() or (len(t.args) == 0 and not t.functor[0].isupper()):
        return t
    return Term(t.functor, [_apply_subst(a, subst) for a in t.args])

def _rename_clause(clause: Clause, prefix: str) -> Clause:
    safe_prefix = prefix.upper() if prefix else "V"
    def rename(t: Term) -> Term:
        if t.is_var():
            return Term(f"{safe_prefix}_{t.functor}")
        return Term(t.functor, [rename(a) for a in t.args])
    return Clause(
        head=rename(clause.head),
        body=[rename(g) for g in clause.body],
        confidence=clause.confidence,
        source=clause.source,
    )

def atom(name: str) -> Term:
    return Term(name.lower())

def var(name: str) -> Term:
    return Term(name.upper())

print("✓ Term and unification defined")

In [ ]:
class KnowledgeBase:
    def __init__(self):
        self.clauses: list[Clause] = []
        self._idx: dict[str, list[int]] = defaultdict(list)

    def assert_clause(self, clause: Clause) -> None:
        idx = len(self.clauses)
        self.clauses.append(clause)
        key = f"{clause.head.functor}/{len(clause.head.args)}"
        self._idx[key].append(idx)

    def matching_clauses(self, goal: Term) -> list[Clause]:
        key = f"{goal.functor}/{len(goal.args)}"
        return [self.clauses[i] for i in self._idx.get(key, [])]

@dataclass
class ProofNode:
    goal: str
    status: str = "pending"
    source: str = "kb"
    confidence: float = 1.0
    children: list["ProofNode"] = field(default_factory=list)

class SLDSolver:
    def __init__(self, kb: KnowledgeBase, document: str, max_depth: int = 8):
        self.kb = kb
        self.document = document
        self.max_depth = max_depth
        self.llm_calls: list[dict] = []
        self._step_counter = 0
        self._grounded: set[str] = set()

    def solve(self, goals: list[Term], subst: Subst, depth: int, trace: ProofNode):
        if depth > self.max_depth or not goals:
            if not goals:
                yield subst, 1.0, trace
            return

        self._step_counter += 1
        if self._step_counter > 500:
            return

        goal = _apply_subst(goals[0], subst)
        rest = goals[1:]
        node = ProofNode(goal=repr(goal))
        trace.children.append(node)

        matched_any = False
        for clause in self.kb.matching_clauses(goal):
            pfx = f"v{depth}_{id(clause) % 1000}"
            renamed = _rename_clause(clause, pfx)
            new_subst = _unify(goal, renamed.head, subst)
            if new_subst is None:
                continue
            matched_any = True
            new_goals = renamed.body + rest
            for sol_subst, sol_conf, sol_trace in self.solve(new_goals, new_subst, depth + 1, node):
                yield sol_subst, sol_conf, sol_trace

        goal_key = repr(goal)
        if not matched_any and goal_key not in self._grounded and self.document:
            self._grounded.add(goal_key)
            response, cost = _llm_call_mock(f"Does {goal} hold?")
            self.llm_calls.append({"predicate": repr(goal), "response": response, "cost": cost})
            if "yes" in response.lower():
                grounded_args = [_apply_subst(a, subst) for a in goal.args]
                self.kb.assert_clause(Clause(head=Term(goal.functor, grounded_args), body=[], source="llm"))
                for clause in self.kb.matching_clauses(goal):
                    new_subst = _unify(goal, clause.head, subst)
                    if new_subst:
                        for sol in self.solve(clause.body + rest, new_subst, depth + 1, node):
                            yield sol

        node.status = "fail"

print("✓ KB and Solver defined")

In [ ]:
def add_rules(kb: KnowledgeBase):
    rules = [
        Clause(Term("parent", [var("X"), var("Y")]), [Term("mother", [var("X"), var("Y")])]),
        Clause(Term("parent", [var("X"), var("Y")]), [Term("father", [var("X"), var("Y")])]),
        Clause(Term("grandparent", [var("X"), var("Z")]),
               [Term("parent", [var("X"), var("Y")]), Term("parent", [var("Y"), var("Z")])]),
        Clause(Term("grandmother", [var("X"), var("Z")]),
               [Term("mother", [var("X"), var("Y")]), Term("parent", [var("Y"), var("Z")])]),
    ]
    for r in rules:
        kb.assert_clause(r)

print("✓ Background rules defined")

In [ ]:
@dataclass
class Task:
    id: str
    document: str
    query_prolog: str
    ground_truth: str

def parse_term(s: str) -> Term | None:
    s = s.strip()
    m = re.match(r"^([a-z_][a-z_0-9]*)\(([^)]+)\)$", s)
    if m:
        functor = m.group(1)
        args = [Term(a.strip()) for a in m.group(2).split(",")]
        return Term(functor, args)
    return Term(s) if s and s[0].islower() else None

def query(kb, doc, goal_str):
    goal = parse_term(goal_str)
    if not goal:
        return None
    solver = SLDSolver(kb, doc, max_depth=MAX_DEPTH)
    root = ProofNode(goal=repr(goal))
    for subst, conf, _ in solver.solve([goal], {}, 0, root):
        result = _apply_subst(goal, subst)
        return {"result": repr(result), "llm_calls": len(solver.llm_calls), "cost": sum(c["cost"] for c in solver.llm_calls)}
    return None

# Demo task
task = Task(
    id="syn_01",
    document="Alice is the mother of Bob. Bob is the father of Charlie.",
    query_prolog="grandmother(alice, charlie)",
    ground_truth="yes"
)

print(f"Task: {task.id}")
print(f"Document: {task.document}")
print(f"Query: {task.query_prolog}")

kb = KnowledgeBase()
add_rules(kb)
kb.assert_clause(Clause(head=Term("mother", [atom("alice"), atom("bob")]), body=[]))
kb.assert_clause(Clause(head=Term("father", [atom("bob"), atom("charlie")]), body=[]))

result = query(kb, task.document, task.query_prolog)
if result:
    print(f"Result: {result['result']}")
    print(f"LLM calls: {result['llm_calls']}")
    print(f"Cost: ${result['cost']:.5f}")
else:
    print("No solution found")

print(f"\nTotal: ${_total_cost_usd:.5f} for {_total_llm_calls} calls")